# Phase 7: Generation Pipeline

**Pipeline**: Vietnamese Financial News RAG System — v3
**Owner**: Member C | **Hardware**: Colab CPU (API calls only)

### What this notebook does
For every item in the **immutable test set** (`qa_pairs_test.parquet`):
1. Retrieve relevant context chunks using the best-performing configuration from Phase 6
2. Build a three-component Vietnamese prompt: [System Instruction] + [Context] + [Question]
3. Generate an answer via `generate_answer()` (Groq → OpenRouter fallback)
4. Save to `evaluation/generation_results.parquet`

> **Model priority** (automatic, handled by `src/generation.py`):
> | Priority | Backend | Model |
> |----------|---------|-------|
> | 1 | Groq LPU | `llama-3.3-70b-versatile` |
> | 2 (fallback) | OpenRouter | `google/gemma-4-31b-it:free` |

> ⚠️ **DO NOT re-run Phase 5** to regenerate the test set. `qa_pairs_test.parquet` is immutable.

## Cell 0 — Environment Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Cloning repo and installing dependencies (first run)...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
        print("Install complete. Restarting kernel to reload packages...")
        os.kill(os.getpid(), 9)  # Force Colab RAM reload
    else:
        print("Repo already exists. Skipping install.")

    # Load API keys from .env stored on Drive
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("\nCell 0 complete.")

## Cell 1 — Imports, Config & API Verification

In [ ]:
import json
import time
import pandas as pd
import faiss
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from src.utils import load_config, resolve_path, ensure_dir, get_env
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever
from src.generation import generate_answer, RAG_SYSTEM_PROMPT

# ── Load config ────────────────────────────────────────────────────────────────
config = load_config()

# ── Generation parameters ──────────────────────────────────────────────────────
# Limit to the 197-item immutable test set (all items if < EVAL_SAMPLE)
EVAL_SAMPLE     = config['evaluation']['eval_sample_size']   # 200
TOP_K           = config['retrieval']['top_k_hybrid']        # 10
CHECKPOINT_EVERY = 20   # Save parquet every N generated answers

# ── Verify at least one API key is present ─────────────────────────────────────
groq_key       = get_env('GROQ_API_KEY')
openrouter_key = get_env('OPENROUTER_API_KEY')
assert groq_key or openrouter_key, "Set GROQ_API_KEY or OPENROUTER_API_KEY before running."

if groq_key:
    print(f"✅ GROQ_API_KEY found  → will use llama-3.3-70b-versatile (Groq LPU)")
else:
    print(f"✅ OPENROUTER_API_KEY found → will use {config['generation']['model']} (OpenRouter)")

# ── Load embedding model for dense retrieval ───────────────────────────────────
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'cpu')
if device == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nLoading embedding model: {model_name} on {device}")
embedding_model = SentenceTransformer(model_name, device=device)
print(f"✅ Embedding model loaded on {embedding_model.device}")

## Cell 2 — Load Immutable Test Set

Loads `qa_pairs_test.parquet` (20% split from Phase 5, **never modify**).
Falls back to `qa_pairs_filtered.parquet` only if test file is absent.

In [ ]:
qa_dir      = resolve_path(config['synthetic_qa'], 'output_dir')
test_path   = os.path.join(qa_dir, 'qa_pairs_test.parquet')
filter_path = os.path.join(qa_dir, 'qa_pairs_filtered.parquet')

if os.path.exists(test_path):
    df_qa = pd.read_parquet(test_path)
    print(f"✅ Loaded immutable test set: {len(df_qa)} QA pairs")
elif os.path.exists(filter_path):
    df_qa = pd.read_parquet(filter_path)
    print(f"⚠️  Test set not found. Using filtered set: {len(df_qa)} QA pairs")
else:
    raise FileNotFoundError(
        f"QA data not found at {test_path}. Ensure Phase 5 has been completed."
    )

# Cap to EVAL_SAMPLE if needed (test set is 197 rows, below the 200 cap)
if len(df_qa) > EVAL_SAMPLE:
    df_eval = df_qa.sample(n=EVAL_SAMPLE, random_state=42).reset_index(drop=True)
    print(f"Sampled {len(df_eval)} / {len(df_qa)} QA pairs (EVAL_SAMPLE={EVAL_SAMPLE})")
else:
    df_eval = df_qa.reset_index(drop=True)
    print(f"Using all {len(df_eval)} QA pairs (below EVAL_SAMPLE cap)")

# Resolve answer column name (Phase 5 may use 'answer' or 'reference_answer')
ANSWER_COL = 'answer' if 'answer' in df_eval.columns else 'reference_answer'
print(f"Ground-truth column: '{ANSWER_COL}'")
df_eval.head(2)

## Cell 3 — Select Best Retrieval Configuration

Reads `evaluation/retrieval_benchmark.csv` from Phase 6 and picks the
configuration with the highest `MRR` score as the single config to use
for generation. This avoids running 9 × 197 = 1,773 LLM calls.

> To override and use a specific config, manually set `BEST_STRATEGY`
> and `BEST_METHOD` in the cell below.

In [ ]:
eval_dir      = resolve_path(config['evaluation'], 'output_dir')
benchmark_csv = os.path.join(eval_dir, 'retrieval_benchmark.csv')

if os.path.exists(benchmark_csv):
    df_bench = pd.read_csv(benchmark_csv)
    # Select configuration with highest MRR
    best_row = df_bench.loc[df_bench['MRR'].idxmax()]
    BEST_STRATEGY = best_row['Strategy']
    BEST_METHOD   = best_row['Method']
    print(f"✅ Best config from Phase 6:")
    print(f"   Strategy : {BEST_STRATEGY}")
    print(f"   Method   : {BEST_METHOD}")
    print(f"   MRR      : {best_row['MRR']:.4f}")
    print(f"   NDCG@10  : {best_row['NDCG@10']:.4f}")
else:
    # Manual fallback — set based on known Phase 6 results
    BEST_STRATEGY = 'fixed_size'
    BEST_METHOD   = 'Hybrid'
    print(f"⚠️  retrieval_benchmark.csv not found.")
    print(f"   Using manual defaults: strategy={BEST_STRATEGY}, method={BEST_METHOD}")

print(f"\nWill generate answers using: [{BEST_STRATEGY}] + [{BEST_METHOD}]")

## Cell 4 — Load Retrieval Indexes for Best Config

In [ ]:
index_base_dir = resolve_path(config['indexing'], 'output_dir')
bm25_base_dir  = resolve_path(config['indexing'], 'bm25_dir')

# ── Dense index ────────────────────────────────────────────────────────────────
faiss_path      = os.path.join(index_base_dir, BEST_STRATEGY, 'index.faiss')
chunk_ids_path  = os.path.join(index_base_dir, BEST_STRATEGY, 'chunk_ids.json')
metadata_path   = os.path.join(index_base_dir, BEST_STRATEGY, 'metadata.parquet')

assert os.path.exists(faiss_path), f"FAISS index not found: {faiss_path}"

faiss_index = faiss.read_index(faiss_path)
with open(chunk_ids_path, 'r', encoding='utf-8') as f:
    chunk_ids = json.load(f)

# Load metadata to map chunk_id → text
df_meta = pd.read_parquet(metadata_path)
chunk_text_map  = dict(zip(df_meta['chunk_id'], df_meta['text']))
chunk_title_map = dict(zip(df_meta['chunk_id'], df_meta.get('title', pd.Series(dtype=str))))

dense_retriever = DenseRetriever(faiss_index, chunk_ids, embedding_model)
print(f"✅ Dense index loaded — {faiss_index.ntotal:,} vectors")

# ── BM25 index ─────────────────────────────────────────────────────────────────
bm25_index, bm25_chunk_ids = load_bm25_index(bm25_base_dir, BEST_STRATEGY)
sparse_retriever = SparseRetriever(bm25_index, bm25_chunk_ids)
print(f"✅ BM25 index loaded  — {len(bm25_chunk_ids):,} chunks")

# ── Hybrid retriever ───────────────────────────────────────────────────────────
hybrid_retriever = HybridRetriever(
    dense_retriever, sparse_retriever,
    rrf_k=config['retrieval']['rrf_k']
)

# Map method name to retriever instance
retriever_map = {
    'Dense':  dense_retriever,
    'Sparse': sparse_retriever,
    'Hybrid': hybrid_retriever,
}
retriever = retriever_map[BEST_METHOD]
print(f"\n✅ Active retriever: {BEST_METHOD}")

## Cell 5 — Generation Loop

Iterates over the evaluation set in batches.
- Retrieves context for each question using the best config
- Builds the three-component Vietnamese prompt
- Calls `generate_answer()` (Groq-first, OpenRouter fallback)
- Checkpoints to `generation_results.parquet` every 20 answers

> ⏱️ **Expected time**: ~5–10 min on Groq (LPU), ~30–60 min on OpenRouter.

In [ ]:
import os
import time
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

ensure_dir(eval_dir)
out_path = os.path.join(eval_dir, 'generation_results.parquet')

# ── Resume from checkpoint if file already exists ─────────────────────────────
if os.path.exists(out_path):
    df_existing = pd.read_parquet(out_path)
    completed_questions = set(df_existing['question'].tolist())
    all_results = df_existing.to_dict('records')
    print(f"🔄 Resuming — {len(all_results)} answers already saved. Skipping completed rows.")
else:
    completed_questions = set()
    all_results = []
    print("🚀 Starting fresh generation run...")

print(f"   Total to generate: {len(df_eval) - len(completed_questions)} remaining\n")

# ── Auto-Fallback Generation Function ──────────────────────────────────────────
def generate_with_auto_fallback(question, contexts):
    # Xây dựng prompt
    prompt = f"{RAG_SYSTEM_PROMPT}\n\nNgữ cảnh:\n" + "\n---\n".join(contexts) + f"\n\nCâu hỏi: {question}"

    # Danh sách các API dự phòng theo thứ tự ưu tiên
    backends = [
        {"name": "Groq", "url": "https://api.groq.com/openai/v1", "key": get_env('GROQ_API_KEY'), "model": "llama-3.3-70b-versatile"},
        {"name": "OpenRouter", "url": "https://openrouter.ai/api/v1", "key": get_env('OPENROUTER_API_KEY'), "model": config['generation']['model']},
        {"name": "Gemini", "url": "https://generativelanguage.googleapis.com/v1beta/openai/", "key": get_env('GEMINI_API_KEY'), "model": "gemini-3.1-flash-lite"}
    ]

    for b in backends:
        if not b["key"]:
            continue

        # Thử tối đa 2 lần cho mỗi API nếu gặp lỗi nghẽn mạng tạm thời (503)
        for attempt in range(2):
            try:
                client = OpenAI(base_url=b["url"], api_key=b["key"])
                response = client.chat.completions.create(
                    model=b["model"],
                    messages=[{"role": "user", "content": prompt}],
                    temperature=config['generation']['temperature']
                )
                return response.choices[0].message.content

            except Exception as e:
                err_str = str(e).lower()
                if "429" in err_str or "rate limit" in err_str:
                    print(f"\n[🔄 FALLBACK] {b['name']} hết hạn mức (429). Đang chuyển sang API tiếp theo...")
                    break # Lỗi 429 thì không thử lại, lập tức sang API khác
                elif "503" in err_str or "unavailable" in err_str:
                    if attempt == 0:
                        print(f"\n[⏳ RETRY] {b['name']} quá tải (503). Đợi 10s để thử lại...")
                        time.sleep(10)
                        continue # Thử lại cùng API
                    else:
                        print(f"\n[🔄 FALLBACK] {b['name']} vẫn bận. Đang chuyển sang API tiếp theo...")
                        break
                else:
                    print(f"\n[⚠️ LỖI] {b['name']} gặp sự cố: {e}")
                    break # Lỗi format/auth thì sang API khác

    # Nếu tất cả API đều thất bại
    raise RuntimeError("Tất cả các API backends đều đã cạn kiệt Rate Limit hoặc quá tải.")

# ── Main generation loop ───────────────────────────────────────────────────────
for i, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Generating answers"):
    question      = row['question']
    ground_truth  = row.get(ANSWER_COL, '')
    doc_id        = row.get('doc_id', '')

    if question in completed_questions:
        continue

    # Step 1: Retrieve relevant context chunks
    retrieved      = retriever.retrieve(question, top_k=TOP_K)
    retrieved_ids  = [cid for cid, _ in retrieved]
    contexts       = [
        chunk_text_map.get(cid, '')
        for cid in retrieved_ids
        if cid in chunk_text_map and chunk_text_map.get(cid, '').strip()
    ]

    retrieved_context = "\n---\n".join(contexts)

    # Step 2: Generate answer with explicit Fallback
    try:
        generated_answer = generate_with_auto_fallback(question, contexts)
    except RuntimeError as re:
        print(f"\n🛑 DỪNG HỆ THỐNG: {re}")
        break # Ngắt vòng lặp ngay lập tức khi cạn sạch API
    except Exception as e:
        print(f"\n[WARNING] Generation failed for row {i}: {e}")
        generated_answer = "[GENERATION_ERROR]"

    # Step 3: Append result
    all_results.append({
        'question':           question,
        'ground_truth':       ground_truth,
        'retrieved_context':  retrieved_context,
        'generated_answer':   generated_answer,
        'doc_id':             doc_id,
        'strategy':           BEST_STRATEGY,
        'method':             BEST_METHOD,
        'retrieved_chunk_ids': json.dumps(retrieved_ids, ensure_ascii=False),
    })
    completed_questions.add(question)

    # Step 4: Checkpoint
    if len(all_results) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_results).to_parquet(out_path, index=False)

    # Thời gian nghỉ giãn cách bắt buộc: 20 giây để hệ thống xả token
    time.sleep(20)

# Save final state before exiting
if all_results:
    pd.DataFrame(all_results).to_parquet(out_path, index=False)
print(f"\nGeneration loop stopped. Total results saved: {len(all_results)}")

In [ ]:
# CELL 5.5: Correct any faulty rows (columns) in the dataset
import pandas as pd
import time
from tqdm import tqdm
import os

# 1. Tải lại file kết quả hiện tại
out_path = os.path.join(eval_dir, 'generation_results.parquet')
df_results = pd.read_parquet(out_path)

# 2. Xác định các chỉ mục (index) chứa lỗi
error_mask = df_results['generated_answer'] == '[GENERATION_ERROR]'
error_indices = df_results[error_mask].index

print(f"🚀 Bắt đầu chạy bù {len(error_indices)} mẫu bị lỗi [GENERATION_ERROR]...\n")

if len(error_indices) > 0:
    # 3. Vòng lặp xử lý lại
    for idx in tqdm(error_indices, desc="Fixing errors"):
        row = df_results.loc[idx]
        question = row['question']

        # Tái sử dụng context đã được retrieve và lưu trong file để tối ưu thời gian
        # Đưa chuỗi string về dạng list 1 phần tử để khớp với đối số của hàm generate_with_auto_fallback
        contexts = [row['retrieved_context']]

        try:
            # Gọi lại cơ chế Fallback (tự động đảo Groq -> OpenRouter -> Gemini)
            new_answer = generate_with_auto_fallback(question, contexts)

            # Ghi đè kết quả mới vào dataframe
            df_results.at[idx, 'generated_answer'] = new_answer
        except Exception as e:
            print(f"\n[⚠️ LỖI] Câu hỏi '{question}' tiếp tục thất bại: {e}")

        # Bắt buộc duy trì cooldown 20 giây để tránh sập Rate Limit lần nữa
        time.sleep(20)

    # 4. Ghi ra một file hoàn toàn mới để tránh xung đột Google Drive
    import time
    timestamp = int(time.time())
    new_fixed_path = os.path.join(eval_dir, f'generation_results_fixed_{timestamp}.parquet')

    df_results.to_parquet(new_fixed_path, index=False)

    # 5. Thống kê lại
    final_errors = len(df_results[df_results['generated_answer'] == '[GENERATION_ERROR]'])
    print(f"\n✅ Quá trình vá lỗi hoàn tất.")
    print(f"✅ ĐÃ LƯU VẬT LÝ TẠI: {new_fixed_path}")
    print(f"📊 Thống kê mới: Thành công {len(df_results) - final_errors}/{len(df_results)} | Lỗi tồn đọng: {final_errors}")

## Cell 6 — Save Final Results & Verify Schema

In [ ]:
# Final save
df_results = pd.DataFrame(all_results)
df_results.to_parquet(out_path, index=False)
print(f"✅ Saved {len(df_results)} generation results → {out_path}")

# ── Verify required output schema ─────────────────────────────────────────────
required_cols = ['question', 'ground_truth', 'retrieved_context', 'generated_answer']
missing = [c for c in required_cols if c not in df_results.columns]
assert not missing, f"Missing required columns: {missing}"
print(f"\n✅ Schema verified. Columns: {df_results.columns.tolist()}")

# ── Summary stats ──────────────────────────────────────────────────────────────
n_errors     = (df_results['generated_answer'] == '[GENERATION_ERROR]').sum()
n_successful = len(df_results) - n_errors
print(f"\nGeneration summary:")
print(f"  Total       : {len(df_results)}")
print(f"  Successful  : {n_successful}")
print(f"  Errors      : {n_errors}")
print(f"  Strategy    : {BEST_STRATEGY}")
print(f"  Method      : {BEST_METHOD}")

# Preview
df_results[['question', 'generated_answer', 'ground_truth']].head(3)

## Cell 7 — Sample Inspection

Display a random sample for quick qualitative review before Phase 8 evaluation.

In [ ]:
import random

df_results = pd.read_parquet(out_path)

# Print 3 random examples in readable format
sample_indices = random.sample(range(len(df_results)), min(3, len(df_results)))

for idx in sample_indices:
    row = df_results.iloc[idx]
    print(f"{'='*70}")
    print(f"Q  : {row['question']}")
    print(f"GT : {str(row['ground_truth'])[:200]}...")
    print(f"GEN: {str(row['generated_answer'])[:300]}...")
    print()

print("Phase 7 complete. Confirm 'Xong' before proceeding to Phase 8 (Evaluation).")